<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session09-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 9: in-class discussion problem (3 of 3)

This is the block 3 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Forget model.eval(), and the answer changes

A network with **dropout** ($p = 0.5$) is trained to tell benign from malignant breast biopsies
(569 biopsies, 30 measurements each). In training mode, dropout switches off a random half of the
hidden units at every forward pass. Then the code predicts the same biopsy five times, *without*
calling `model.eval()`. What does it print?

**Fix and run.** The cell trains the network in a second or two, then predicts. Run it as it is and
look at the five predictions. The line marked `FIX` puts the model in the wrong mode for predicting.
Fix it, and run the cell again.

In [1]:
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)      # y = 1: benign
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_tr)
X_tr = torch.tensor(scaler.transform(X_tr), dtype=torch.float32)
X_te = torch.tensor(scaler.transform(X_te), dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.float32)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(30, 64), nn.ReLU(), nn.Dropout(p=0.5), nn.Linear(64, 1))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()
for epoch in range(200):
    opt.zero_grad()
    loss = nn.functional.binary_cross_entropy_with_logits(model(X_tr).squeeze(1), y_tr)
    loss.backward()
    opt.step()

model.train()                    # FIX: the wrong mode for predicting
torch.manual_seed(1)
biopsy = X_te[6:7]               # one test biopsy
with torch.no_grad():
    for i in range(5):
        print(f"prediction {i + 1}: P(benign) = {torch.sigmoid(model(biopsy)).item():.2f}")

prediction 1: P(benign) = 0.27
prediction 2: P(benign) = 0.57
prediction 3: P(benign) = 0.04
prediction 4: P(benign) = 0.59
prediction 5: P(benign) = 0.31


**If you finish early: L1 or L2?** Block 3 also penalised large weights. Ten features, only
the first three matter. Which penalty sets the useless weights to exactly zero? Predict, then run.

In [2]:
import numpy as np
from sklearn.linear_model import Lasso, Ridge

rng = np.random.default_rng(0)
Xs = rng.normal(size=(200, 10))
ys = Xs @ np.array([2.0, -1.5, 1.0, 0, 0, 0, 0, 0, 0, 0]) + 0.1 * rng.normal(size=200)
print("L1 (Lasso):", np.round(Lasso(alpha=0.1).fit(Xs, ys).coef_, 3))
print("L2 (Ridge):", np.round(Ridge(alpha=10).fit(Xs, ys).coef_, 3))

L1 (Lasso): [ 1.897 -1.382  0.873  0.    -0.    -0.     0.     0.    -0.    -0.   ]
L2 (Ridge): [ 1.907 -1.406  0.936  0.022 -0.018 -0.024  0.012  0.007 -0.008 -0.019]


<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the model was still in training mode, so dropout switched off
a different random half of the hidden units at every call. The same biopsy got five different
predictions, some above 0.5 and some below: benign one moment, malignant the next. The fix is

```python
model.eval()
```

which switches dropout off (and uses all units, scaled to match training). Then the five
predictions are identical, P(benign) = 0.31. Training does not change the weights during
prediction; dropout does change which units take part. The same holds for batch normalisation, which
uses the batch statistics in training mode and the stored averages in evaluation mode. So every
training loop in this course calls `model.train()` before training and `model.eval()` before
evaluating.

In the L1/L2 cell, Lasso (L1) sets all seven useless weights to exactly 0, while Ridge (L2) only
shrinks them: small, but not zero. L1 selects features; L2 keeps them all.

</details>